In [1]:
%reload_ext autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
from evaluate import load
bleu = load('sacrebleu')

/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [3]:
import IPython.display as ipd
import whisper
import sys
sys.path.append("/home/romolo/VT1/coqui-tts")
from model_conf import ModelPaths, load_tts_and_trainer

/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


In [4]:
paths = ModelPaths()
tts, model, train_model, config = load_tts_and_trainer(paths)

 > Using model: xtts
>> DVAE weights restored from: /home/romolo/VT1/coqui-tts/XTTS_v2.0_original_model_files/dvae.pth


In [5]:
asr_model = whisper.load_model("base")

In [6]:
import os

In [7]:
from TTS.tts.models.xtts import load_audio
import soundfile as sf

In [8]:
from huggingface_hub import hf_hub_download

# automatically checks for cached file, optionally set `cache_dir` location
model_file = hf_hub_download(repo_id='Jenthe/ECAPA2', filename='ecapa2.pt', cache_dir=None)

In [9]:
import torch
import torchaudio
import torch.nn.functional as F

ecapa2 = torch.jit.load(model_file, map_location='cuda')


In [10]:
import pandas as pd

In [11]:
df = pd.read_csv('/data/dev/metadata/0000.csv')
df.head()

,id,orig_path,start,end,text,speaker_id,book_id,segment_id,dataset,split,store_id,estimated_size_bytes
0,10362_11341_000000,http://www.archive.org/download/glinni_sacri_1...,195.34,206.13,tanto d'ogni laudato esser la prima di dio la ...,10362,11341,0,mls_italian,dev,mls_italian_dev_10362_11341_000000,951678.0
1,10362_11341_000001,http://www.archive.org/download/glinni_sacri_1...,42.59,54.46,qual angolo ti raccogliea nascente quando il t...,10362,11341,1,mls_italian,dev,mls_italian_dev_10362_11341_000001,1046934.0
2,10362_11341_000002,http://www.archive.org/download/glinni_sacri_1...,164.34,179.28,un estranio giovinetto si posò sul monumento e...,10362,11341,2,mls_italian,dev,mls_italian_dev_10362_11341_000002,1317708.0
3,10362_11341_000003,http://www.archive.org/download/glinni_sacri_1...,183.58,196.81,come vittima innanzi all'altar non lo seppe il...,10362,11341,3,mls_italian,dev,mls_italian_dev_10362_11341_000003,1166886.0
4,10362_11341_000004,http://www.archive.org/download/glinni_sacri_1...,179.31,195.34,anco ogni giorno se ne parla e tanto secol vi ...,10362,11341,4,mls_italian,dev,mls_italian_dev_10362_11341_000004,1413846.0


In [12]:
hdf5path = "/data/dev/audios/0000.hdf5"

In [13]:
df.store_id.iloc[11000]

'mls_english_dev_10326_10194_000016'

In [14]:
df.store_id.iloc[11100]

'mls_english_dev_8862_10240_000002'

In [15]:
from TTS.tts.models.xtts import load_audio

In [16]:
target = load_audio(df.store_id.iloc[11002],22050,hdf5path)
ipd.Audio(target,rate=22050)

In [17]:
refrence = load_audio(df.store_id.iloc[11100],22050,hdf5path)
ipd.Audio(refrence,rate=22050)

In [18]:
data = model.prep_batch(
    lang='en', 
    text=df.text[11002],
    target_sample=df.store_id[11002],
    ref_sample=df.store_id.iloc[11100],
    train_model=train_model,
    max_conditioning_length=2000,
    min_conditioning_length=5000,
    target_hdf5_path=hdf5path,
    ref_hdf5_path=hdf5path,
    target_sample_rate=22050
)

In [19]:
output = model.forward_iteration_hdf5_with_batch(
    langs=["en","en",'en'], 
    texts=df.text[11002:11005].to_list(),
    target_samples=df.store_id[11002:11005].to_list(),
    ref_samples=[df.store_id.iloc[11100]]*3,
    train_model=train_model,
    max_conditioning_length=config.model_args.max_conditioning_length, 
    min_conditioning_length=config.model_args.min_conditioning_length,
    tts = tts,
    ecapa = ecapa2,
    asr_model =asr_model,
    target_hdf5_path=hdf5path,
    ref_hdf5_path=hdf5path)

Sample 0, Iteration 0: BLUE: 100.00000000000004, WER: 0.0, Quality: 0.9817716376855969
Sample 1, Iteration 0: BLUE: 67.46950518256601, WER: 0.19444444444444445, Quality: 0.9295524748870068
Sample 2, Iteration 0: BLUE: 74.66745587002926, WER: 0.14285714285714285, Quality: 0.678410893040044
Sample 0, Iteration 1: BLUE: 94.1435681721808, WER: 0.022727272727272728, Quality: 0.9584700727665967
Sample 1, Iteration 1: BLUE: 63.76897100442937, WER: 0.2222222222222222, Quality: 0.9257662030350831
Sample 2, Iteration 1: BLUE: 65.9506725604744, WER: 0.21428571428571427, Quality: 0.7301293356078011
Sample 0, Iteration 2: BLUE: 94.0028651976138, WER: 0.022727272727272728, Quality: 0.9701688039878553
Sample 1, Iteration 2: BLUE: 70.39758268589667, WER: 0.19444444444444445, Quality: 0.9261752691947751
Sample 2, Iteration 2: BLUE: 63.594237815417394, WER: 0.30952380952380953, Quality: 0.743066902316752
Sample 0, Iteration 3: BLUE: 91.84043388013012, WER: 0.045454545454545456, Quality: 0.99223177401687

In [20]:
df.text[11002:11005].to_list()

['to make the people love who hate me now my dreams are over i have ceased to cry against the fate that made men love my mouth and left their spirits all too deaf to hear the little songs that echoed through my soul',
 'they shall behold each one his dream that fashions me anew with hair like lakes that glint beneath the stars dark as sweet midnight or with hair aglow like burnished gold that still retains the fire',
 'castell shook his head impatiently ask the question man if you will but never take the answer if it is against you wait rather and ask it again and went on peter without noticing his grey eyes lighting with a sudden fire']

In [21]:
ipd.Audio(output[3][0][2], rate=24000)

In [22]:
df.text.iloc[11004]

'castell shook his head impatiently ask the question man if you will but never take the answer if it is against you wait rather and ask it again and went on peter without noticing his grey eyes lighting with a sudden fire'

In [23]:
df.store_id.iloc[11100:11110].to_list()

['mls_english_dev_8862_10240_000002',
 'mls_english_dev_10326_10244_000000',
 'mls_english_dev_10326_10244_000001',
 'mls_english_dev_10326_10244_000002',
 'mls_english_dev_10326_10244_000003',
 'mls_english_dev_10326_10244_000004',
 'mls_english_dev_10326_10244_000005',
 'mls_english_dev_10326_10244_000006',
 'mls_english_dev_10326_10244_000007',
 'mls_english_dev_10326_10244_000008']

In [24]:
df.speaker_id.iloc[10000]

137

In [25]:
df.speaker_id.iloc[11100:11110]

11100     8862
11101    10326
11102    10326
11103    10326
11104    10326
11105    10326
11106    10326
11107    10326
11108    10326
11109    10326
Name: speaker_id, dtype: int64

In [26]:
df.speaker_id.iloc[11090:11100].to_list()

[9679, 9679, 9679, 9679, 9679, 9679, 9679, 9679, 8862, 8862]

In [27]:
df.text.iloc[10900]

'one reason he looks so steadily at you that you think he is studying you is because the light is so strong in the daytime that his sight is bad but the owl is not as wise as he is said to be he does some foolish things as well as other birds in fact he is sometimes more foolish than any other bird would be in the same place'

In [28]:
output1 = model.forward_iteration_hdf5('en',
                                       df.text.iloc[10980],
                                       df.store_id.iloc[10980],
                                       df.store_id.iloc[11111],
                                       train_model,
                                       config.model_args.max_conditioning_length, 
                                       config.model_args.min_conditioning_length,
                                       tts,ecapa2,
                                       asr_model,target_hdf5_path=hdf5path,ref_hdf5_path=hdf5path)

Iteration 0: BLUE: 90.61874434879648, WER: 0.034482758620689655, Target Cosine: tensor([0.3106], device='cuda:0'), Reference Cosine: tensor([0.4393], device='cuda:0')
Iteration 1: BLUE: 78.11895757488891, WER: 0.10344827586206896, Target Cosine: tensor([0.2865], device='cuda:0'), Reference Cosine: tensor([0.4630], device='cuda:0')
Iteration 2: BLUE: 90.61874434879648, WER: 0.034482758620689655, Target Cosine: tensor([0.2339], device='cuda:0'), Reference Cosine: tensor([0.4423], device='cuda:0')
Iteration 3: BLUE: 86.9639866212288, WER: 0.06896551724137931, Target Cosine: tensor([0.2368], device='cuda:0'), Reference Cosine: tensor([0.4529], device='cuda:0')
Iteration 4: BLUE: 84.21464365949143, WER: 0.10344827586206896, Target Cosine: tensor([0.2407], device='cuda:0'), Reference Cosine: tensor([0.4555], device='cuda:0')
Cleaned up temp HDF5 file: /home/romolo/VT1/coqui-tts/development/temp/iter_temp_nxgr1exu.hdf5


In [29]:
ipd.Audio(output1[3][4], rate=24000)

In [30]:
import h5py

In [31]:
# Save HDF5 audio data to WAV files
with h5py.File(hdf5path, 'r') as f:
    # Get a sample audio from the HDF5 file
    audio_data = f[df.store_id.iloc[11010]][:]
    audio_data2 = f[df.store_id.iloc[11100]][:]
    
    # Save as WAV file
    sf.write('/tmp/sample_audio.wav', audio_data, 22050)
    sf.write('/tmp/sample_audio2.wav', audio_data2, 22050)
    
print("Audio saved to /tmp/sample_audio.wav")

Audio saved to /tmp/sample_audio.wav


In [32]:
audio_data

array([5.4121017e-05, 6.0743219e-05, 7.7950870e-05, ..., 1.4407569e-04,
       1.5962022e-04, 5.0211915e-05], dtype=float32)

In [33]:
model.forward_iteration("en",
                        df.text.iloc[11010],
                        '/tmp/sample_audio.wav',
                        ['/tmp/sample_audio2.wav'],
                        train_model,
                        config.model_args.max_conditioning_length, 
                        config.model_args.min_conditioning_length,
                        tts,ecapa2,
                        asr_model)

Iteration 0: BLUE: 53.4156557789369, WER: 0.25, Target Cosine: tensor([0.0880], device='cuda:0'), Reference Cosine: tensor([0.3715], device='cuda:0')
Iteration 1: BLUE: 49.25224642477487, WER: 0.28125, Target Cosine: tensor([0.0735], device='cuda:0'), Reference Cosine: tensor([0.4069], device='cuda:0')
Iteration 2: BLUE: 56.24819984534316, WER: 0.21875, Target Cosine: tensor([0.0245], device='cuda:0'), Reference Cosine: tensor([0.3816], device='cuda:0')
Iteration 3: BLUE: 47.65277227474963, WER: 0.3125, Target Cosine: tensor([-0.0172], device='cuda:0'), Reference Cosine: tensor([0.3589], device='cuda:0')
Iteration 4: BLUE: 46.66388090961478, WER: 0.375, Target Cosine: tensor([-0.0333], device='cuda:0'), Reference Cosine: tensor([0.3493], device='cuda:0')


({'wav_0': '/home/romolo/VT1/coqui-tts/data/outputs/models/iterate_output_0.wav',
  'wav_1': '/home/romolo/VT1/coqui-tts/data/outputs/models/iterate_output_1.wav',
  'wav_2': '/home/romolo/VT1/coqui-tts/data/outputs/models/iterate_output_2.wav',
  'wav_3': '/home/romolo/VT1/coqui-tts/data/outputs/models/iterate_output_3.wav',
  'wav_4': '/home/romolo/VT1/coqui-tts/data/outputs/models/iterate_output_4.wav'},
 {'blue_scores': [53.4156557789369,
   49.25224642477487,
   56.24819984534316,
   47.65277227474963,
   46.66388090961478],
  'wer_scores': [0.25, 0.28125, 0.21875, 0.3125, 0.375],
  'target_cosine_similarities': [tensor([0.0880], device='cuda:0'),
   tensor([0.0735], device='cuda:0'),
   tensor([0.0245], device='cuda:0'),
   tensor([-0.0172], device='cuda:0'),
   tensor([-0.0333], device='cuda:0')],
  'reference_cosine_similarities': [tensor([0.3715], device='cuda:0'),
   tensor([0.4069], device='cuda:0'),
   tensor([0.3816], device='cuda:0'),
   tensor([0.3589], device='cuda:0'),

In [34]:
output1[1]

{'blue_scores': [90.61874434879648,
  78.11895757488891,
  90.61874434879648,
  86.9639866212288,
  84.21464365949143],
 'wer_scores': [0.034482758620689655,
  0.10344827586206896,
  0.034482758620689655,
  0.06896551724137931,
  0.10344827586206896],
 'target_cosine_similarities': [tensor([0.3106], device='cuda:0'),
  tensor([0.2865], device='cuda:0'),
  tensor([0.2339], device='cuda:0'),
  tensor([0.2368], device='cuda:0'),
  tensor([0.2407], device='cuda:0')],
 'reference_cosine_similarities': [tensor([0.4393], device='cuda:0'),
  tensor([0.4630], device='cuda:0'),
  tensor([0.4423], device='cuda:0'),
  tensor([0.4529], device='cuda:0'),
  tensor([0.4555], device='cuda:0')]}

In [35]:
output1

({},
 {'blue_scores': [90.61874434879648,
   78.11895757488891,
   90.61874434879648,
   86.9639866212288,
   84.21464365949143],
  'wer_scores': [0.034482758620689655,
   0.10344827586206896,
   0.034482758620689655,
   0.06896551724137931,
   0.10344827586206896],
  'target_cosine_similarities': [tensor([0.3106], device='cuda:0'),
   tensor([0.2865], device='cuda:0'),
   tensor([0.2339], device='cuda:0'),
   tensor([0.2368], device='cuda:0'),
   tensor([0.2407], device='cuda:0')],
  'reference_cosine_similarities': [tensor([0.4393], device='cuda:0'),
   tensor([0.4630], device='cuda:0'),
   tensor([0.4423], device='cuda:0'),
   tensor([0.4529], device='cuda:0'),
   tensor([0.4555], device='cuda:0')]},
 [0.8274621963500977,
  0.805030107498169,
  0.8658075332641602,
  0.847103476524353,
  0.8279293179512024],
 [array([ 8.4142451e-04,  3.1273911e-04,  6.8811775e-04, ...,
         -7.3909003e-05,  2.7255662e-04,  2.7095768e-04], dtype=float32),
  array([ 8.8003767e-04,  2.9081962e-04,  

In [36]:
output1[1]

{'blue_scores': [90.61874434879648,
  78.11895757488891,
  90.61874434879648,
  86.9639866212288,
  84.21464365949143],
 'wer_scores': [0.034482758620689655,
  0.10344827586206896,
  0.034482758620689655,
  0.06896551724137931,
  0.10344827586206896],
 'target_cosine_similarities': [tensor([0.3106], device='cuda:0'),
  tensor([0.2865], device='cuda:0'),
  tensor([0.2339], device='cuda:0'),
  tensor([0.2368], device='cuda:0'),
  tensor([0.2407], device='cuda:0')],
 'reference_cosine_similarities': [tensor([0.4393], device='cuda:0'),
  tensor([0.4630], device='cuda:0'),
  tensor([0.4423], device='cuda:0'),
  tensor([0.4529], device='cuda:0'),
  tensor([0.4555], device='cuda:0')]}

In [37]:
ipd.Audio(output1[3][0],rate=24000)

In [38]:
import h5py

In [39]:
orig_target_sample = '/home/romolo/VT1/coqui-tts/test_data/Dataset/references_new/EN_1624/1624-142933-0000.wav'
ref_samples = ['/home/romolo/VT1/coqui-tts/test_data/Dataset/references_new/EN_1447/1447-17506-0000.wav']

# Create HDF5 file and save audio
hdf5_path = '/tmp/test_audio.hdf5'
with h5py.File(hdf5_path, 'w') as f:
    # Save target audio with store_id = 0
    audio_target, sr_target = torchaudio.load(orig_target_sample)
    print(audio_target.shape,sr_target)
    # Resample to 22050 Hz (standard sample rate for storage)
    if sr_target != 22050:
        audio_target = torchaudio.functional.resample(audio_target, orig_freq=sr_target, new_freq=22050)
    f.create_dataset('0', data=audio_target.numpy())
    
    # Save ref audio with store_id = 1
    audio_ref, sr_ref = torchaudio.load(ref_samples[0])
    print(audio_ref.shape,sr_ref)
    if sr_ref != 22050:
        audio_ref = torchaudio.functional.resample(audio_ref, orig_freq=sr_ref, new_freq=22050)
    f.create_dataset('1', data=audio_ref.numpy())

print(f"Audio files saved to {hdf5_path}")

torch.Size([1, 67694]) 22050
torch.Size([1, 70671]) 22050
Audio files saved to /tmp/test_audio.hdf5


In [40]:
text = asr_model.transcribe(orig_target_sample)["text"]
print(text)

 Peter sees Rosebrest and finds Redcoat.


In [41]:
output1 = model.forward_iteration_hdf5(
    'en',
    text,
    "0",
    ["1"],
    train_model,
    config.model_args.max_conditioning_length, 
    config.model_args.min_conditioning_length,
    tts,ecapa2,
    asr_model,target_hdf5_path=hdf5_path,ref_hdf5_path=hdf5_path
    )


Iteration 0: BLUE: 100.00000000000004, WER: 0.0, Target Cosine: tensor([0.0886], device='cuda:0'), Reference Cosine: tensor([0.4197], device='cuda:0')
Iteration 1: BLUE: 100.00000000000004, WER: 0.0, Target Cosine: tensor([0.0692], device='cuda:0'), Reference Cosine: tensor([0.4867], device='cuda:0')
Iteration 2: BLUE: 100.00000000000004, WER: 0.0, Target Cosine: tensor([0.0857], device='cuda:0'), Reference Cosine: tensor([0.4605], device='cuda:0')
Iteration 3: BLUE: 75.98356856515926, WER: 0.16666666666666666, Target Cosine: tensor([0.0358], device='cuda:0'), Reference Cosine: tensor([0.4690], device='cuda:0')
Iteration 4: BLUE: 17.965205598154213, WER: 0.5, Target Cosine: tensor([0.0113], device='cuda:0'), Reference Cosine: tensor([0.4649], device='cuda:0')
Cleaned up temp HDF5 file: /home/romolo/VT1/coqui-tts/development/temp/iter_temp_9ocettqq.hdf5


In [42]:
output1

({},
 {'blue_scores': [100.00000000000004,
   100.00000000000004,
   100.00000000000004,
   75.98356856515926,
   17.965205598154213],
  'wer_scores': [0.0, 0.0, 0.0, 0.16666666666666666, 0.5],
  'target_cosine_similarities': [tensor([0.0886], device='cuda:0'),
   tensor([0.0692], device='cuda:0'),
   tensor([0.0857], device='cuda:0'),
   tensor([0.0358], device='cuda:0'),
   tensor([0.0113], device='cuda:0')],
  'reference_cosine_similarities': [tensor([0.4197], device='cuda:0'),
   tensor([0.4867], device='cuda:0'),
   tensor([0.4605], device='cuda:0'),
   tensor([0.4690], device='cuda:0'),
   tensor([0.4649], device='cuda:0')]},
 [0.9557235240936279,
  0.9653953313827515,
  0.9571586847305298,
  0.8987575769424438,
  0.7443282008171082],
 [array([ 1.9078440e-04, -3.1596047e-04, -2.1355505e-04, ...,
          6.9054928e-05,  3.5356574e-05,  1.2338703e-04], dtype=float32),
  array([ 1.7087853e-04, -4.5485151e-04, -4.2357092e-04, ...,
          2.5740417e-05, -2.2771823e-05,  4.0565672

In [43]:
orig_target_sample

'/home/romolo/VT1/coqui-tts/test_data/Dataset/references_new/EN_1624/1624-142933-0000.wav'

In [44]:
ref_samples

['/home/romolo/VT1/coqui-tts/test_data/Dataset/references_new/EN_1447/1447-17506-0000.wav']

In [45]:

orig_target_sample = '/home/romolo/VT1/coqui-tts/test_data/Dataset/references_new/EN_1624/1624-142933-0000.wav'
ref_samples = ['/home/romolo/VT1/coqui-tts/test_data/Dataset/references_new/EN_1447/1447-17506-0000.wav']

In [46]:
output = model.forward_iteration('en',text,orig_target_sample,ref_samples,train_model,config.model_args.max_conditioning_length, config.model_args.min_conditioning_length,tts,ecapa2,asr_model)

Iteration 0: BLUE: 100.00000000000004, WER: 0.0, Target Cosine: tensor([-0.0086], device='cuda:0'), Reference Cosine: tensor([0.1886], device='cuda:0')
Iteration 1: BLUE: 14.25876976452075, WER: 0.6666666666666666, Target Cosine: tensor([0.0103], device='cuda:0'), Reference Cosine: tensor([0.3318], device='cuda:0')
Iteration 2: BLUE: 100.00000000000004, WER: 0.0, Target Cosine: tensor([-0.0657], device='cuda:0'), Reference Cosine: tensor([0.3776], device='cuda:0')
Iteration 3: BLUE: 100.00000000000004, WER: 0.0, Target Cosine: tensor([-0.0431], device='cuda:0'), Reference Cosine: tensor([0.3550], device='cuda:0')
Iteration 4: BLUE: 14.25876976452075, WER: 0.6666666666666666, Target Cosine: tensor([-0.0818], device='cuda:0'), Reference Cosine: tensor([0.3369], device='cuda:0')


In [47]:
output1[1]

{'blue_scores': [100.00000000000004,
  100.00000000000004,
  100.00000000000004,
  75.98356856515926,
  17.965205598154213],
 'wer_scores': [0.0, 0.0, 0.0, 0.16666666666666666, 0.5],
 'target_cosine_similarities': [tensor([0.0886], device='cuda:0'),
  tensor([0.0692], device='cuda:0'),
  tensor([0.0857], device='cuda:0'),
  tensor([0.0358], device='cuda:0'),
  tensor([0.0113], device='cuda:0')],
 'reference_cosine_similarities': [tensor([0.4197], device='cuda:0'),
  tensor([0.4867], device='cuda:0'),
  tensor([0.4605], device='cuda:0'),
  tensor([0.4690], device='cuda:0'),
  tensor([0.4649], device='cuda:0')]}

In [48]:
output1[1]

{'blue_scores': [100.00000000000004,
  100.00000000000004,
  100.00000000000004,
  75.98356856515926,
  17.965205598154213],
 'wer_scores': [0.0, 0.0, 0.0, 0.16666666666666666, 0.5],
 'target_cosine_similarities': [tensor([0.0886], device='cuda:0'),
  tensor([0.0692], device='cuda:0'),
  tensor([0.0857], device='cuda:0'),
  tensor([0.0358], device='cuda:0'),
  tensor([0.0113], device='cuda:0')],
 'reference_cosine_similarities': [tensor([0.4197], device='cuda:0'),
  tensor([0.4867], device='cuda:0'),
  tensor([0.4605], device='cuda:0'),
  tensor([0.4690], device='cuda:0'),
  tensor([0.4649], device='cuda:0')]}

In [49]:
ipd.Audio(output1[3][0],rate=24000)

In [50]:
output1 = model.forward_iteration_hdf5('en',df.text.iloc[11000],df.store_id.iloc[11000],df.store_id.iloc[11100],train_model,config.model_args.max_conditioning_length, config.model_args.min_conditioning_length,tts,ecapa2,asr_model,target_hdf5_path=hdf5path,ref_hdf5_path=hdf5path)

Iteration 0: BLUE: 77.68557860134777, WER: 0.1, Target Cosine: tensor([0.0183], device='cuda:0'), Reference Cosine: tensor([0.3824], device='cuda:0')
Iteration 1: BLUE: 93.36510695862633, WER: 0.025, Target Cosine: tensor([0.0521], device='cuda:0'), Reference Cosine: tensor([0.4348], device='cuda:0')
Iteration 2: BLUE: 70.76054943226818, WER: 0.125, Target Cosine: tensor([0.0496], device='cuda:0'), Reference Cosine: tensor([0.4668], device='cuda:0')
Iteration 3: BLUE: 66.0397975565253, WER: 0.175, Target Cosine: tensor([0.0599], device='cuda:0'), Reference Cosine: tensor([0.5148], device='cuda:0')
Iteration 4: BLUE: 70.85382291457624, WER: 0.125, Target Cosine: tensor([0.0718], device='cuda:0'), Reference Cosine: tensor([0.5289], device='cuda:0')
Cleaned up temp HDF5 file: /home/romolo/VT1/coqui-tts/development/temp/iter_temp_ucp7nzeu.hdf5


In [51]:
ipd.Audio(output1[3][3],rate=24000)

In [52]:

orig_target_sample = '/home/romolo/VT1/coqui-tts/test_data/Dataset/references_new/EN_1624/1624-142933-0000.wav'
ref_samples = ['/home/romolo/VT1/coqui-tts/test_data/Dataset/references_new/EN_1447/1447-17506-0000.wav']

In [53]:
text = asr_model.transcribe(orig_target_sample)["text"]

In [54]:
output = model.forward_iteration('en',text,orig_target_sample,ref_samples,train_model,config.model_args.max_conditioning_length, config.model_args.min_conditioning_length,tts,ecapa2,asr_model)

Iteration 0: BLUE: 100.00000000000004, WER: 0.0, Target Cosine: tensor([-0.0086], device='cuda:0'), Reference Cosine: tensor([0.1886], device='cuda:0')
Iteration 1: BLUE: 14.25876976452075, WER: 0.6666666666666666, Target Cosine: tensor([0.0103], device='cuda:0'), Reference Cosine: tensor([0.3318], device='cuda:0')
Iteration 2: BLUE: 100.00000000000004, WER: 0.0, Target Cosine: tensor([-0.0657], device='cuda:0'), Reference Cosine: tensor([0.3776], device='cuda:0')
Iteration 3: BLUE: 100.00000000000004, WER: 0.0, Target Cosine: tensor([-0.0431], device='cuda:0'), Reference Cosine: tensor([0.3550], device='cuda:0')
Iteration 4: BLUE: 14.25876976452075, WER: 0.6666666666666666, Target Cosine: tensor([-0.0818], device='cuda:0'), Reference Cosine: tensor([0.3369], device='cuda:0')


In [55]:
ipd.Audio(output[3][2],rate=20400)

In [56]:
from run_test import segment_quality_score

In [57]:
tar_audio, sr = torchaudio.load(orig_target_sample) # sample rate of 16 kHz expected
tar_audio = torchaudio.functional.resample(tar_audio, orig_freq=sr, new_freq=16_000)
tar_embedding = ecapa2(tar_audio.to('cuda'))

ref_audio, sr2 = torchaudio.load(ref_samples[0]) # sample rate of 16 kHz expected
ref_audio = torchaudio.functional.resample(ref_audio, orig_freq=sr2, new_freq=16_000)
ref_embedding = ecapa2(ref_audio.to('cuda'))

In [58]:
scores, info = segment_quality_score(asr_model, tar_embedding, ref_embedding,output[3][0],ecapa2)

In [59]:
info

{0: {'overall_quality': 0.519412424787879,
  'wer': 0.0,
  'blue': 1.0,
  'target_sim': 0.010083328932523727,
  'ref_sim': 0.20406237244606018},
 1: {'overall_quality': 0.4250415533781051,
  'wer': 1.0,
  'blue': 0,
  'target_sim': -0.08810341358184814,
  'ref_sim': 0.16435088217258453}}

In [60]:
def calc_sim(aud1,aud2):
    #cossim between embeddings
    audio, sr = torchaudio.load(aud1)
    audio = torchaudio.functional.resample(audio, orig_freq=sr, new_freq=16_000)# sample rate of 16 kHz expected
    embedding = ecapa2(audio.to('cuda'))
    ref_audio, sr = torchaudio.load(aud2) # sample rate of 16 kHz expected
    ref_audio = torchaudio.functional.resample(ref_audio, orig_freq=sr, new_freq=16_000)
    ref_embedding = ecapa2(ref_audio.to('cuda'))
    sim = F.cosine_similarity(embedding, ref_embedding)
    return sim

In [61]:
for i in range(0,10):
    targ = calc_sim(f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav","/home/romolo/VT1/coqui-tts/data/target.wav")
    print(f"sample {i}")
    print(f"sim -- target to output: {targ}")
    ref = calc_sim(f"/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_{i}.wav",ref_samples[0])
    print(f"sim -- ref to output: {ref}")


RuntimeError: Failed to open the input "/home/romolo/VT1/coqui-tts/data/outputs/iterate_output_0.wav" (No such file or directory).
Exception raised from get_input_format_context at /__w/audio/audio/pytorch/audio/src/libtorio/ffmpeg/stream_reader/stream_reader.cpp:42 (most recent call first):
frame #0: c10::Error::Error(c10::SourceLocation, std::string) + 0x96 (0x7fb963f6c1b6 in /home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/torch/lib/libc10.so)
frame #1: c10::detail::torchCheckFail(char const*, char const*, unsigned int, std::string const&) + 0x64 (0x7fb963f15a76 in /home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/torch/lib/libc10.so)
frame #2: <unknown function> + 0x42034 (0x7fb80944e034 in /home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/torio/lib/libtorio_ffmpeg4.so)
frame #3: torio::io::StreamingMediaDecoder::StreamingMediaDecoder(std::string const&, std::optional<std::string> const&, std::optional<std::map<std::string, std::string, std::les